# Exercises XP – Week 5, Day 3: Air Traffic Data Analysis
### Inferential Statistics and Regression Analysis

## Section 1: Setup and Data Loading
The course didn't provide an `air_traffic_data.csv` file. As the instructions say, if the file is not found, the notebook **creates sample data** with the same columns: monthly US-style air traffic from 2003 to 2023, with realistic trends, seasonality and a COVID-19 drop in 2020. If you later get the real file, put it next to the notebook and it will be used instead.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler

def make_sample_data(seed=42):
    rng = np.random.default_rng(seed)
    dates = pd.date_range("2003-01-01", "2023-09-01", freq="MS")
    t = np.arange(len(dates))
    season = 1 + 0.10 * np.sin(2 * np.pi * (dates.month - 4) / 12)       # summer peak
    trend = 1 + 0.012 * t / 12                                            # slow growth per year
    covid = np.where((dates >= "2020-04-01") & (dates < "2021-06-01"), 0.35, 1.0)
    covid = np.where((dates >= "2020-03-01") & (dates < "2020-04-01"), 0.7, covid)
    noise = lambda s: rng.normal(1, s, len(dates))

    dom_pax = 50_000_000 * trend * season * covid * noise(0.03)
    int_pax = 7_000_000 * (trend ** 1.6) * season * np.where(covid < 1, covid * 0.7, 1) * noise(0.04)
    dom_flt = dom_pax / (78 + 0.6 * t / 12) * noise(0.02)                  # planes get fuller over time
    int_flt = int_pax / (155 + 0.5 * t / 12) * noise(0.03)
    dom_rpm = dom_pax * rng.normal(900, 25, len(dates))                   # ~900 miles per domestic trip

    df = pd.DataFrame({"Year": dates.year, "Month": dates.month,
                       "Dom_Pax": dom_pax, "Int_Pax": int_pax, "Dom_Flt": dom_flt, "Int_Flt": int_flt,
                       "Dom_RPM": dom_rpm}).round(0).astype("int64")
    df["Pax"] = df["Dom_Pax"] + df["Int_Pax"]
    df["Flt"] = df["Dom_Flt"] + df["Int_Flt"]
    return df

if os.path.exists("air_traffic_data.csv"):
    df = pd.read_csv("air_traffic_data.csv")
    print("Loaded the real air traffic file.")
else:
    df = make_sample_data()
    print("air_traffic_data.csv not found – using generated sample data.")

print("Shape:", df.shape)

## Section 2: Exploratory Data Analysis (EDA)

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
df.describe().round(0)

In [ ]:
print("Missing values per column:")
df.isnull().sum()

In [ ]:
num_cols = ["Dom_Pax", "Int_Pax", "Pax", "Dom_Flt", "Int_Flt", "Flt", "Dom_RPM"]
corr = df[num_cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation matrix")
plt.show()

In [ ]:
# Strongest correlations (each pair once, excluding a variable with itself)
pairs = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack().sort_values(ascending=False)
print("Strongest correlations:")
print(pairs.head(8).round(3))

**Observations:** almost every pair has |r| > 0.7. Some of these are **built in**: `Pax` is the sum of `Dom_Pax` and `Int_Pax`, `Flt` is the sum of the flight columns, and `Dom_RPM` is passengers × distance. So their very high correlations (~0.99) are expected, not discoveries. This also warns us about **multicollinearity** when using several of them together in a regression.

## Section 3: Hypothesis Testing
Significance level **α = 0.05**.

### Test 1: Domestic vs. international passengers
- **H₀:** mean domestic passengers = mean international passengers
- **H₁:** the means are different

In [ ]:
t_stat, p_value = stats.ttest_ind(df["Dom_Pax"], df["Int_Pax"], equal_var=False)
print(f"Mean domestic passengers per month:      {df['Dom_Pax'].mean():,.0f}")
print(f"Mean international passengers per month: {df['Int_Pax'].mean():,.0f}")
print(f"t = {t_stat:.2f}, p-value = {p_value:.3g}")
print("Reject H0" if p_value < 0.05 else "Fail to reject H0")

### Test 2: Is the correlation between total passengers and total flights significant?
- **H₀:** no correlation (ρ = 0)
- **H₁:** a significant correlation exists (ρ ≠ 0)

In [ ]:
r, p_corr = stats.pearsonr(df["Pax"], df["Flt"])
print(f"Pearson r = {r:.3f}, p-value = {p_corr:.3g}")
print("Reject H0" if p_corr < 0.05 else "Fail to reject H0")

**Interpretation:**
- **Test 1:** p is essentially 0 (far below 0.05), so we **reject H₀**. Domestic traffic is much larger than international traffic: roughly **54 million vs. 8 million passengers per month**, about 6–7 times more. The difference is statistically **and** practically significant.
- **Test 2:** r ≈ **0.97** with p ≈ 0, so we **reject H₀**. There is a very strong, positive and significant correlation: months with more flights carry more passengers. Correlation is not causation, though. Both are driven by demand (season, economy, COVID-19), and airlines schedule flights to match it.

## Section 4: Simple Linear Regression
Predict **total passengers (Pax)** from **total flights (Flt)**.

In [ ]:
X_simple = df[["Flt"]]
y = df["Pax"]

X_train_s, X_test_s, y_train, y_test = train_test_split(X_simple, y, test_size=0.2, random_state=42)

simple_model = LinearRegression()
simple_model.fit(X_train_s, y_train)
y_pred_simple = simple_model.predict(X_test_s)

r2_s = r2_score(y_test, y_pred_simple)
mse_s = mean_squared_error(y_test, y_pred_simple)
rmse_s = np.sqrt(mse_s)
mae_s = mean_absolute_error(y_test, y_pred_simple)

print(f"Equation: Pax = {simple_model.intercept_:,.0f} + {simple_model.coef_[0]:.2f} × Flt")
print(f"R² = {r2_s:.4f} | MSE = {mse_s:,.0f} | RMSE = {rmse_s:,.0f} | MAE = {mae_s:,.0f}")

In [ ]:
residuals_s = y_test - y_pred_simple

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(X_test_s, y_test, alpha=0.6, label="Actual")
line_x = np.linspace(df["Flt"].min(), df["Flt"].max(), 100).reshape(-1, 1)
axes[0].plot(line_x, simple_model.predict(pd.DataFrame(line_x, columns=["Flt"])), "r-", label="Regression line")
axes[0].set_xlabel("Total flights")
axes[0].set_ylabel("Total passengers")
axes[0].set_title("Simple linear regression")
axes[0].legend()

axes[1].scatter(y_pred_simple, residuals_s, alpha=0.6)
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_xlabel("Predicted passengers")
axes[1].set_ylabel("Residual (actual − predicted)")
axes[1].set_title("Residual plot – simple model")
plt.tight_layout()
plt.show()

## Section 5: Multiple Linear Regression
Features: `Dom_Pax`, `Int_Pax`, `Dom_Flt`, `Int_Flt`, `Dom_RPM` (avoiding `Pax`, the target, and `Flt`, the sum of the flight columns). Scaled with `StandardScaler`, **fitted on the training data only**.

⚠️ Note: `Pax` is *exactly* `Dom_Pax + Int_Pax`, so with these features the model can reconstruct the target almost perfectly. That's why R² will be ~1. It's a useful lesson in **target leakage**. I also add a second, more realistic model that only uses **flights and RPM** (no passenger counts).

In [ ]:
features_multi = ["Dom_Pax", "Int_Pax", "Dom_Flt", "Int_Flt", "Dom_RPM"]

def fit_multi(features):
    X_tr, X_te, y_tr, y_te = train_test_split(df[features], y, test_size=0.2, random_state=42)
    scaler = StandardScaler()
    X_tr_sc = scaler.fit_transform(X_tr)       # fit only on training data
    X_te_sc = scaler.transform(X_te)
    model = LinearRegression().fit(X_tr_sc, y_tr)
    pred = model.predict(X_te_sc)
    metrics = {"R²": r2_score(y_te, pred), "RMSE": np.sqrt(mean_squared_error(y_te, pred)),
               "MAE": mean_absolute_error(y_te, pred)}
    return model, pred, y_te, metrics

multi_model, y_pred_multi, y_test_m, m_multi = fit_multi(features_multi)
print("Multiple regression (course feature set):", {k: round(float(v), 4) for k, v in m_multi.items()})
print("Standardized coefficients:")
print(pd.Series(multi_model.coef_, index=features_multi).round(0))

no_leak = ["Dom_Flt", "Int_Flt", "Dom_RPM"]
nl_model, y_pred_nl, _, m_nl = fit_multi(no_leak)
print("\nMultiple regression without passenger columns:", {k: round(float(v), 4) for k, v in m_nl.items()})

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_pred_nl, y_test_m - y_pred_nl, alpha=0.6, color="seagreen")
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Predicted passengers")
plt.ylabel("Residual")
plt.title("Residual plot – multiple regression (flights + RPM)")
plt.show()

## Section 6: Model Comparison and Analysis

In [ ]:
comparison = pd.DataFrame({
    "Simple (Flt)": {"R²": r2_s, "RMSE": rmse_s, "MAE": mae_s},
    "Multiple (course features)": m_multi,
    "Multiple (flights + RPM)": m_nl,
}).T
print(comparison.round(4))

for name in ["Multiple (course features)", "Multiple (flights + RPM)"]:
    r2_imp = (comparison.loc[name, "R²"] - r2_s) / r2_s * 100
    rmse_imp = (comparison.loc[name, "RMSE"] - rmse_s) / rmse_s * 100
    print(f"\n{name} vs simple: R² change {r2_imp:+.2f}% | RMSE change {rmse_imp:+.1f}%")

best = comparison["R²"].idxmax()
print("\nBest model by R²:", best)

## Section 7: Statistical Insights and Conclusions
1. **Hypothesis tests:** domestic and international passenger volumes are significantly different (domestic ≈ 6–7× larger), and total passengers and total flights are strongly and significantly correlated (r ≈ 0.97).
2. **Simple regression (Pax ~ Flt):** R² ≈ **0.89**, so flights alone explain about 89% of the variation in passengers. Each extra flight adds about **90 passengers** on average (the average plane load). The residual plot shows structure, though: the number of passengers per flight changes over time (planes get fuller), which a single straight line can't capture.
3. **Multiple regression (course feature set):** R² = **1.00**, a "perfect" model. This is not real predictive power: `Pax` is literally `Dom_Pax + Int_Pax`, both of which are inputs. This is **target leakage**, and the model just learned to add them up (all other coefficients are ~0).
4. **Multiple regression (flights + RPM only):** R² ≈ **0.96**, RMSE about **43% lower** than the simple model. It's the most honest "better" model: separating domestic and international flights and adding miles flown captures more of how passengers relate to capacity.
5. **Correlations:** almost everything moves together with overall demand; the very highest correlations (0.99+) come from columns that are sums or products of each other.

**Recommendations for airlines:**
- Use **flight-based models** to forecast passenger volumes and plan staff, gates and catering, but update them regularly, because passengers per flight change over time.
- Plan domestic and international operations **separately**: they differ hugely in size and react differently to shocks (international traffic recovered more slowly after COVID-19 in this data).
- Track **load factor** (passengers per flight) as a KPI. The drift in residuals shows it's a key driver.
- When building models, **remove derived columns** (totals, sums) from the features to avoid misleadingly perfect results.

*(Note: these numbers come from the generated sample data, so they illustrate the method; with the real file the exact values would change.)*

## Section 8: Reflection Questions
**1. What do the hypothesis test results reveal about air traffic patterns?**
Domestic travel dominates air traffic, and passenger numbers and flight numbers rise and fall together. The network is demand-driven: when demand drops (e.g. COVID-19), both collapse at the same time.

**2. Why did one regression model perform better than the other?**
The multiple regression uses more information: domestic and international flights separately (which carry very different numbers of passengers) plus miles flown. With the course feature set it performed "perfectly" because the target is just the sum of two inputs (leakage). The flights + RPM model shows the honest improvement over using total flights alone.

**3. How can airlines use correlation insights operationally?**
The strong flight–passenger link lets airlines translate schedules into expected passenger volumes (and vice versa) for capacity planning, pricing and airport staffing. Changes in that relationship (passengers per flight) are an early signal that planes are too full or too empty.

**4. What do residual plots tell you about model assumptions?**
Linear regression assumes residuals are random, centred on 0, with constant spread. The simple model's residuals show patterns and uneven spread, so a single straight line misses part of the structure (e.g. the change in load factor over time). The multiple model's residuals are smaller and closer to random, so its assumptions fit better.

**5. What are practical applications of these statistical models?**
Forecasting passenger demand, planning fleet and staff, setting budgets, testing whether a change (new route, price change, crisis) had a significant effect, and creating KPIs and alerts when actual traffic deviates from what the model expects.